# Task 13: A/B Testing and Hypothesis Testing

**Author:** Arshpreet Singh

This notebook validates whether website Variant B improves conversion rate compared with Variant A using a two-proportion z-test and a 95% confidence interval.

## Hypotheses

- **Null hypothesis (H0):** Conversion rate for Group B is equal to Group A.
- **Alternative hypothesis (H1):** Conversion rate for Group B is different from Group A.
- **Significance level:** alpha = 0.05.

In [5]:
import math
import os
import tempfile
from pathlib import Path

import pandas as pd
from scipy import stats

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
mpl_config = Path(tempfile.gettempdir()) / "ab_testing_mplconfig"
mpl_config.mkdir(exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(mpl_config))

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

try:
    from statsmodels.stats.proportion import proportions_ztest
except Exception:
    proportions_ztest = None

DATA_PATH = ROOT / "data" / "website_traffic_data.csv"

df = pd.read_csv(DATA_PATH)
df.head()

,visitor_id,visit_date,group,device,traffic_channel,session_minutes,pages_viewed,converted,revenue_usd
0,VIS-00001,2026-05-27,A,Mobile,Paid Search,4.79,4,0,0.0
1,VIS-00002,2026-05-26,A,Desktop,Organic,5.39,4,0,0.0
2,VIS-00003,2026-05-25,A,Mobile,Referral,4.78,5,0,0.0
3,VIS-00004,2026-05-26,A,Desktop,Organic,7.50,3,0,0.0
4,VIS-00005,2026-05-03,A,Desktop,Email,3.33,3,0,0.0


In [6]:
summary = (
    df.groupby("group")
    .agg(
        visitors=("visitor_id", "count"),
        conversions=("converted", "sum"),
        conversion_rate=("converted", "mean"),
        revenue_usd=("revenue_usd", "sum"),
    )
    .sort_index()
)
summary

,visitors,conversions,conversion_rate,revenue_usd
group,,,,
A,1200,148,0.123333,8522.00
B,1200,186,0.155000,11560.11


In [7]:
x_a = int(summary.loc["A", "conversions"])
x_b = int(summary.loc["B", "conversions"])
n_a = int(summary.loc["A", "visitors"])
n_b = int(summary.loc["B", "visitors"])
p_a = x_a / n_a
p_b = x_b / n_b

if proportions_ztest:
    z_stat, p_value = proportions_ztest([x_b, x_a], [n_b, n_a], alternative="two-sided")
else:
    pooled = (x_a + x_b) / (n_a + n_b)
    se_pooled = math.sqrt(pooled * (1 - pooled) * (1 / n_a + 1 / n_b))
    z_stat = (p_b - p_a) / se_pooled
    p_value = 2 * (1 - stats.norm.cdf(abs(z_stat)))

absolute_lift = p_b - p_a
relative_lift = absolute_lift / p_a
se_unpooled = math.sqrt(p_a * (1 - p_a) / n_a + p_b * (1 - p_b) / n_b)
ci_low = absolute_lift - 1.96 * se_unpooled
ci_high = absolute_lift + 1.96 * se_unpooled
decision = "Reject H0" if p_value < 0.05 else "Fail to reject H0"

print(f"Group A conversion rate: {p_a:.2%}")
print(f"Group B conversion rate: {p_b:.2%}")
print(f"Absolute lift: {absolute_lift:.2%}")
print(f"Relative lift: {relative_lift:.2%}")
print(f"Z-statistic: {z_stat:.4f}")
print(f"P-value: {p_value:.6f}")
print(f"95% CI for B - A: [{ci_low:.2%}, {ci_high:.2%}]")
print(f"Decision: {decision}")

Group A conversion rate: 12.33%
Group B conversion rate: 15.50%
Absolute lift: 3.17%
Relative lift: 25.68%
Z-statistic: 2.2410
P-value: 0.025023
95% CI for B - A: [0.40%, 5.93%]
Decision: Reject H0


In [8]:
ax = (summary["conversion_rate"] * 100).plot(
    kind="bar",
    color=["#3b6ea8", "#2f8f6f"],
    figsize=(7, 4),
    rot=0,
)
ax.set_title("Website A/B Test Conversion Rate")
ax.set_xlabel("Experiment Group")
ax.set_ylabel("Conversion Rate (%)")
for i, value in enumerate(summary["conversion_rate"] * 100):
    ax.text(i, value + 0.3, f"{value:.2f}%", ha="center")
plt.tight_layout()
ax.figure

<Figure size 700x400 with 1 Axes>

## Conclusion

Group B produced a conversion-rate lift of **3.17%** over Group A. The p-value was **0.025023**, so the decision is **Reject H0** at alpha = 0.05. The 95% confidence interval for the difference B - A is **[0.40%, 5.93%]**.